In [1]:
from utils_task2 import RedditLinkDataset

dataset = RedditLinkDataset(path="Reddit")
train_loader, val_loader, test_loader = dataset.get_link_loaders(batch_size=1024)

Archi totali: 57307946
Archi positivi in Train: 80231126
Coppie di archi pos/neg in Val: 11461588
Coppie di archi pos/neg in Test: 22923178


In [2]:
lr = 1e-4
num_epochs = 10

In [3]:
from torch_geometric.nn import GCNConv
import torch
import torch.nn.functional as F


class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # Encoder: Estrae le feature topologiche dei nodi dal grafo rimanente
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        
        # Decoder: Prende la concatenazione degli embedding di due nodi e calcola la probabilità di legame
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels * 2, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(hidden_channels, 1) # Output singolo logit (0 o 1)
        )

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        # Estrazione degli indici dei nodi sorgente e destinazione della coppia
        nodes_src = edge_label_index[0]
        nodes_dst = edge_label_index[1]
        
        # Recupero dei rispettivi embedding generati dall'encoder
        z_src = z[nodes_src]
        z_dst = z[nodes_dst]
        
        # Concatenazione delle caratteristiche dei due nodi affiancate
        coppie = torch.cat([z_src, z_dst], dim=-1)
        return self.mlp(coppie).view(-1)

    def forward(self, x, edge_index, edge_label_index):
        z = self.encode(x, edge_index)
        return self.decode(z, edge_label_index)

In [4]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nInizio Task 2 (Link Prediction) su dispositivo: {device}")

num_features = dataset.dataset.num_features

# Inizializzazione Rete, Ottimizzatore e Loss Binaria
model = GCNLinkPredictor(in_channels=num_features, hidden_channels=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
criterion = torch.nn.BCEWithLogitsLoss() # Ottimale per la classificazione binaria da logits

# Scaler moderno per gestire il mixed precision senza azzerare i gradienti
scaler = torch.amp.GradScaler('cuda')

# Funzione interna di train per epoca
def train_epoch(epoch_idx):
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoca {epoch_idx:02d} [Train Link]", leave=False)
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Esecuzione in Mixed Precision (AMP) per proteggere la VRAM
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            # L'encoder usa batch.edge_index, il decoder predice su batch.edge_label_index
            out = model(batch.x, batch.edge_index, batch.edge_label_index)
            loss = criterion(out, batch.edge_label)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * batch.edge_label.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    return total_loss / dataset.train_set.edge_label_index.size(1)

# Funzione interna di valutazione (Calcolo AUC-ROC e AP richiesti)
@torch.no_grad()
def evaluate(loader, desc="Valutazione Link"):
    model.eval()
    all_preds = []
    all_targets = []
    
    for batch in tqdm(loader, desc=desc, leave=False):
        batch = batch.to(device)
        
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            out = model(batch.x, batch.edge_index, batch.edge_label_index)
            # Applichiamo la sigmoide per trasformare i logit in probabilità pure [0, 1]
            probs = torch.sigmoid(out)
            
        all_preds.append(probs.cpu())
        all_targets.append(batch.edge_label.cpu())
        
    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_true = torch.cat(all_targets, dim=0).numpy()
    
    # Estrazione delle metriche strutturali richieste dal progetto
    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    return auc, ap

# Loop principale delle epoche (Impostato a 5 per via della mole di archi da macinare)
print("\n--- AVVIO LOOP DI ADDESTRAMENTO LINK PREDICTION ---")
for epoch in range(1, num_epochs + 1):
    loss_t = train_epoch(epoch)
    val_auc, val_ap = evaluate(val_loader, desc=f"Epoca {epoch:02d} [Val Link]")
    print(f"Epoca: {epoch:02d}/{num_epochs:02d} | Loss Train: {loss_t:.4f} | Val AUC-ROC: {val_auc:.4f} | Val AP: {val_ap:.4f}")

# Verifica finale sul test set blindato
print("\n--- VERIFICA FINALE SUL TEST SET ARCHI ---")
test_auc, test_ap = evaluate(test_loader, desc="Test Finale")
print(f"\n[RISULTATI GLOBALI TASK 2]")
print(f"-> Area Under the ROC Curve (AUC-ROC): {test_auc:.4f}")
print(f"-> Average Precision (AP Score): {test_ap:.4f}\n")


Inizio Task 2 (Link Prediction) su dispositivo: cuda

--- AVVIO LOOP DI ADDESTRAMENTO LINK PREDICTION ---


KeyboardInterrupt: 